# 04 · Zero-shot Chronos-Bolt

First foundation-model test: `amazon/chronos-bolt-mini`, no training, on the locked validation set.

**Key results**
- 28-day context: MAE 7.34 MU — beats XGBoost at day 1 but falls behind from day 2.
- Context length matters: 56 days is best (6.14 MU on the 176 origins that support 84 days).
- On those same 176 origins XGBoost still wins (4.94 MU). Saved as the paired evaluation manifest `gridpulse_eval_176_v1.parquet`.

> Paths are relative to `notebooks/`. Outputs are from the original research run; see [`data/README.md`](../data/README.md) for data that must be downloaded.

➡ Next: [05 · Zero-shot TimesFM 2.5](05_zero_shot_timesfm.ipynb)

## Add dependencies

Run once in a terminal: `uv add chronos-forecasting torch`

Load Chronos locally

In [2]:
import torch
from chronos import BaseChronosPipeline

MODEL_ID = "amazon/chronos-bolt-mini"

pipeline = BaseChronosPipeline.from_pretrained(
    MODEL_ID,
    device_map="cpu",
    torch_dtype=torch.float32,
)

print("Chronos model loaded:", MODEL_ID)

Chronos model loaded: amazon/chronos-bolt-mini


Load our locked validation dataset

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("../data")

features = pd.read_parquet(
    DATA / "processed/gridpulse_features_v1.parquet"
)

# The v1 parquet contains the corrected,
# non-crossing chronological splits.
val = features.query(
    "split == 'validation'"
).sort_values(["origin", "horizon"])

origins = val["origin"].drop_duplicates()

print("Validation origins:", len(origins))
print("Validation predictions:", len(val))

assert len(origins) == 257
assert len(val) == 1799

Validation origins: 257
Validation predictions: 1799


Run one Chronos forecast

In [5]:
first_origin = origins.iloc[0]

sample = val[
    val["origin"] == first_origin
].sort_values("horizon")

# Oldest observation first, newest last
context = np.array(
    [
        sample.iloc[0][f"lag_{lag}"]
        for lag in range(28, 0, -1)
    ],
    dtype=np.float32,
)

with torch.inference_mode():
    quantiles, mean = pipeline.predict_quantiles(
        inputs=torch.tensor(context).unsqueeze(0),
        prediction_length=7,
        quantile_levels=[0.1, 0.5, 0.9],
    )

print("Quantile shape:", quantiles.shape)
print("Mean shape:", mean.shape)

print("\nActual:")
print(sample["target"].to_numpy())

print("\nChronos median forecast:")
print(quantiles[0, :, 1].cpu().numpy())

Quantile shape: torch.Size([1, 7, 3])
Mean shape: torch.Size([1, 7])

Actual:
[70.5 66.3 71.8 70.  73.4 71.2 72. ]

Chronos median forecast:
[71.665054 70.908646 71.086    71.39513  71.187485 71.032196 70.82873 ]


Batched Chronos inference

In [6]:
import numpy as np
import torch
import pandas as pd

HORIZON = 7
CONTEXT = 28
BATCH_SIZE = 32

val = val.sort_values(["origin", "horizon"]).copy()

# One history sequence per forecast origin
origin_rows = val.drop_duplicates("origin").sort_values("origin")

lag_cols = [f"lag_{i}" for i in range(CONTEXT, 0, -1)]

contexts = origin_rows[lag_cols].to_numpy(dtype=np.float32)

print("Contexts shape:", contexts.shape)
assert contexts.shape == (257, 28)

predictions = []

for start in range(0, len(contexts), BATCH_SIZE):
    batch = torch.tensor(contexts[start:start+BATCH_SIZE])

    with torch.inference_mode():
        quantiles, _ = pipeline.predict_quantiles(
            inputs=batch,
            prediction_length=HORIZON,
            quantile_levels=[0.1, 0.5, 0.9],
        )

    median = quantiles[:, :, 1].cpu().numpy()
    predictions.append(median)

predictions = np.concatenate(predictions, axis=0)

print("Predictions shape:", predictions.shape)
assert predictions.shape == (257, 7)

Contexts shape: (257, 28)
Predictions shape: (257, 7)


Align forecasts and evaluate

In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

chronos_results = pd.DataFrame({
    "origin": np.repeat(origin_rows["origin"].to_numpy(), HORIZON),
    "horizon": np.tile(np.arange(1, HORIZON + 1), len(origin_rows)),
    "chronos_pred": predictions.reshape(-1),
})

comparison = val[
    ["origin", "target_date", "horizon", "target"]
].merge(
    chronos_results,
    on=["origin", "horizon"],
    validate="one_to_one",
)

assert len(comparison) == 1799
assert comparison["chronos_pred"].notna().all()

actual = comparison["target"].to_numpy()
predicted = comparison["chronos_pred"].to_numpy()

mae = mean_absolute_error(actual, predicted)
rmse = np.sqrt(mean_squared_error(actual, predicted))
wape = 100 * np.abs(actual - predicted).sum() / np.abs(actual).sum()

print(f"Chronos validation MAE:  {mae:.3f} MU")
print(f"Chronos validation RMSE: {rmse:.3f} MU")
print(f"Chronos validation WAPE: {wape:.2f}%")

Chronos validation MAE:  7.336 MU
Chronos validation RMSE: 9.884 MU
Chronos validation WAPE: 8.03%


Compare models by horizon

In [8]:
chronos_horizon = (
    comparison.assign(
        abs_error=lambda d: (
            d["target"] - d["chronos_pred"]
        ).abs()
    )
    .groupby("horizon")["abs_error"]
    .mean()
)

xgb_horizon = {
    1: 4.741,
    2: 5.328,
    3: 5.686,
    4: 5.889,
    5: 6.089,
    6: 6.301,
    7: 6.535,
}

weekly_horizon = {
    1: 8.138,
    2: 8.132,
    3: 8.092,
    4: 7.950,
    5: 7.799,
    6: 7.685,
    7: 7.626,
}

horizon_results = pd.DataFrame({
    "Chronos": chronos_horizon,
    "XGBoost": pd.Series(xgb_horizon),
    "Weekly Naive": pd.Series(weekly_horizon),
})

horizon_results.index.name = "horizon"

print(horizon_results.round(3).to_string())

         Chronos  XGBoost  Weekly Naive
horizon                                
1          4.569    4.741         8.138
2          6.565    5.328         8.132
3          7.476    5.686         8.092
4          7.800    5.889         7.950
5          8.166    6.089         7.799
6          8.297    6.301         7.685
7          8.477    6.535         7.626


Check longer-context eligibility

In [9]:
# Check which existing validation origins support
# complete 28-, 56-, and 84-day contexts.

energy = pd.read_csv(
    DATA / "processed/delhi_energy_clean.csv",
    parse_dates=["Date"],
    index_col="Date",
).sort_index()["energy_met_MU"]

context_lengths = [28, 56, 84]

eligibility = []

for origin in origins:
    row = {"origin": origin}

    for length in context_lengths:
        history = energy.loc[
            origin - pd.Timedelta(days=length - 1):origin
        ]

        row[f"context_{length}"] = (
            len(history) == length
            and history.notna().all()
        )

    eligibility.append(row)

eligibility = pd.DataFrame(eligibility)

print("Eligible validation origins:")
for length in context_lengths:
    print(
        f"{length} days:",
        eligibility[f"context_{length}"].sum()
    )

print(
    "\nOrigins with all 84 days:",
    eligibility["context_84"].sum()
)

Eligible validation origins:
28 days: 257
56 days: 211
84 days: 176

Origins with all 84 days: 176


Run paired Chronos inference

In [ ]:
import numpy as np
import pandas as pd
import torch

# Lock identical evaluation origins.
common_origins = (
    eligibility.loc[eligibility["context_84"], "origin"]
    .sort_values()
    .tolist()
)

assert len(common_origins) == 176

HORIZON = 7
BATCH_SIZE = 32

# Get actual future values for each origin.
actuals = (
    val[val["origin"].isin(common_origins)]
    .sort_values(["origin", "horizon"])
    ["target"]
    .to_numpy()
    .reshape(-1, HORIZON)
)

results = {}

for length in [28, 56, 84]:
    contexts = np.stack([
        energy.loc[
            origin - pd.Timedelta(days=length - 1):origin
        ].to_numpy(dtype=np.float32)
        for origin in common_origins
    ])

    batches = []

    for start in range(0, len(contexts), BATCH_SIZE):
        batch = torch.tensor(
            contexts[start:start + BATCH_SIZE]
        )

        with torch.inference_mode():
            quantiles, _ = pipeline.predict_quantiles(
                inputs=batch,
                prediction_length=HORIZON,
                quantile_levels=[0.1, 0.5, 0.9],
            )

        batches.append(
            quantiles[:, :, 1].cpu().numpy()
        )

    predictions = np.concatenate(batches, axis=0)

    assert predictions.shape == actuals.shape

    errors = np.abs(actuals - predictions)

    results[length] = {
        "MAE": errors.mean(),
        "RMSE": np.sqrt(
            np.mean((actuals - predictions) ** 2)
        ),
        "Day_1_MAE": errors[:, 0].mean(),
        "Day_7_MAE": errors[:, 6].mean(),
    }

context_comparison = pd.DataFrame(results).T
context_comparison.index.name = "Context_days"

display(context_comparison.round(3))

,MAE,RMSE,Day_1_MAE,Day_7_MAE
Context_days,,,,
28,6.800,9.409,4.091,8.319
56,6.135,8.692,3.628,7.772
84,6.176,8.969,3.535,7.987


Paired XGBoost comparison

In [12]:
xgb_results = pd.read_parquet(
    DATA / "processed/xgb_validation_v01.parquet"
)

common = xgb_results[
    xgb_results["origin"].isin(common_origins)
].copy()

xgb_mae = np.abs(
    common["target"] - common["xgb_pred"]
).mean()

print(f"XGBoost MAE on 176 origins: {xgb_mae:.3f} MU")
print(f"Chronos-56 MAE on 176 origins: {context_comparison.loc[56, 'MAE']:.3f} MU")

XGBoost MAE on 176 origins: 4.939 MU
Chronos-56 MAE on 176 origins: 6.135 MU


Save the Manifest

In [13]:
manifest = (
    val[val["origin"].isin(common_origins)]
    [["origin", "target_date", "horizon", "target"]]
    .sort_values(["origin", "horizon"])
    .reset_index(drop=True)
)

assert len(manifest) == 176 * 7
assert manifest["origin"].nunique() == 176
assert manifest.groupby("origin").size().eq(7).all()

path = DATA / "processed/gridpulse_eval_176_v1.parquet"
manifest.to_parquet(path, index=False)

print("Saved:", path)
print("Shape:", manifest.shape)

Saved: ..\data\processed\gridpulse_eval_176_v1.parquet
Shape: (1232, 4)
